In [1]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough
)
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from typing import Tuple, List
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
import os
from langchain_community.graphs import Neo4jGraph
from langchain.text_splitter import TokenTextSplitter
from langchain_experimental.graph_transformers import LLMGraphTransformer
from neo4j import GraphDatabase
from yfiles_jupyter_graphs import GraphWidget
from langchain_community.vectorstores import Neo4jVector
from langchain_community.vectorstores.neo4j_vector import remove_lucene_chars
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
import json
import networkx as nx

/Users/willikneer/dev/Uni/group-4/mc3/backend/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3670: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [32]:
graph = Neo4jGraph(
    url="bolt://localhost:7687/",
    username="neo4j",
    password="ava25-DB!!"
)
llm = Ollama(
            model="llama3.1",
            base_url="https://ollama.joos.dbvis.de",
        )


In [3]:
default_cypher = "MATCH (s)-[r]->(t) RETURN s,r,t LIMIT 250"

def showGraph(cypher: str = default_cypher):
    driver = GraphDatabase.driver(
        "bolt://localhost:7687/",
        auth=("neo4j","ava25-DB!!")
    )
    session = driver.session()
    result = session.run(cypher)
    graph = result.graph()
    widget = GraphWidget(graph=graph)
    display(widget)

showGraph()

GraphWidget(layout=Layout(height='800px', width='100%'))

In [4]:
vector_index = Neo4jVector.from_existing_graph(
    
)

TypeError: Neo4jVector.from_existing_graph() missing 4 required positional arguments: 'embedding', 'node_label', 'embedding_node_property', and 'text_node_properties'

In [5]:
# Lösche den ersten Fulltext-Index
graph.query("DROP INDEX index_d1ebd962")

# Lösche den zweiten Fulltext-Index
graph.query("DROP INDEX index_de1a2369")

DatabaseError: {code: Neo.DatabaseError.Schema.IndexDropFailed} {message: Unable to drop index called `index_d1ebd962`. There is no such index.}

In [6]:
graph.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (n:Entity) ON EACH [n.id]"
)

[]

In [7]:

class Entities(BaseModel):
    names: List[str] = Field(
        ...,
        description="All the person, locations that appear in the database"
    )

prompt_entities = ChatPromptTemplate.from_messages([
    ("system", 
     "You are an information extraction assistant. Your task is to extract all PERSON names and LOCATION names from the given text. "
     "Combine both types into a single list called 'names'. "
     "Return only a JSON object like this:\n"
     "\"names\": [\"Barack Obama\", \"California\"]\n"
     "Do not separate persons and locations. Only return valid JSON."),
    ("human", "Text: {input}")
])


def extract_entities(text: str) -> Entities:
    formatted_prompt = prompt_entities.format(input=text)
    response = llm.invoke(formatted_prompt)
    print(response)
    try:
        data = json.loads(response)
        return Entities(**data)
    except Exception as e:
        print("Parsing failed:", e)
        raise

entities = extract_entities("Who is Nadja Conti?")
print("Extracted:", entities.names)

{
  "names": ["Nadja Conti"]
}
Extracted: ['Nadja Conti']


In [ ]:
from typing import Dict


class Node(BaseModel):
    id: str
    labels: List[str]
    properties: Dict[str, object]

class Relationship(BaseModel):
    type: str
    source: str  
    target: str  
    properties: Dict[str, object]

class PathResult(BaseModel):
    nodes: List[Node]
    relationships: List[Relationship]

def generate_full_text_query(input: str) -> str:

    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

def structured_retriever(question: str) -> List[PathResult]:
    result = []
    entities = extract_entities(question)
    
    for entity in entities.names:
        response = graph.query(
            """
            CALL db.index.fulltext.queryNodes('entity', $query, {limit:2})
            YIELD node AS start_entity
            MATCH path = (start_entity)-[rels*..5]-(end_entity:Entity)
            WHERE all(n IN nodes(path)[1..-1] WHERE NOT n:Entity)
            AND start_entity <> end_entity
            RETURN {
                nodes: [n IN nodes(path) | {
                    id: n.id, 
                    labels: labels(n), 
                    properties: properties(n)
                }],
                relationships: [r IN rels | {
                    type: type(r),
                    source: startNode(r).id,
                    target: endNode(r).id,
                    properties: properties(r)
                }]
            } AS path_data
            """,
            {"query": generate_full_text_query(entity)},
        )
        
        for record in response:
            path_data = record["path_data"]
            
            nodes = [
                Node(
                    id=node["id"],
                    labels=node["labels"],
                    properties=node["properties"]
                ) for node in path_data["nodes"]
            ]
            
            relationships = [
                Relationship(
                    type=rel["type"],
                    source=rel["source"],
                    target=rel["target"],
                    properties=rel["properties"]
                ) for rel in path_data["relationships"]
            ]
            
            result.append(PathResult(
                nodes=nodes,
                relationships=relationships
            ))
    
    return result



In [28]:
def describe_path(path: PathResult) -> str:
    node_map = {node.id: node for node in path.nodes}

    # 1. Beschreibung für Knoten
    node_descriptions = []
    for node in path.nodes:
        label = node.labels[0] if node.labels else "Node"
        name = node.properties.get("name") or node.properties.get("title") or node.id
        props = ", ".join(f"{k}: {v}" for k, v in node.properties.items() if k not in ["name", "title"])
        desc = f"{name} ({label})"
        if props:
            desc += f" [{props}]"
        node_descriptions.append(desc)

    # 2. Beschreibung für Kanten
    edge_descriptions = []
    for rel in path.relationships:
        source_node = node_map.get(rel.source)
        target_node = node_map.get(rel.target)

        if source_node and target_node:
            source_name = source_node.properties.get("name") or source_node.id
            target_name = target_node.properties.get("name") or target_node.id
            rel_type = rel.type
            props = ", ".join(f"{k}: {v}" for k, v in rel.properties.items())
            edge_text = f"{source_name} -[{rel_type}]-> {target_name}"
            if props:
                edge_text += f" [{props}]"
            edge_descriptions.append(edge_text)

    full_description = "Nodes:\n" + "\n".join(node_descriptions)
    full_description += "\n\Relationships:\n" + "\n".join(edge_descriptions)
    return full_description


def summarize_and_score_path(path: PathResult, question: str) -> Dict[str, object]:
    path_desc = describe_path(path)

    prompt = f"""
        You are given a path from a knowledge graph and a user question.
        Summarize the path in natural language and assess whether it is relevant to answering the user's question.

        Question: {question}

        Path:
        {path_desc}

        Just Answer with a JSON Object with the following structure:
        {{
        "summary": "<natural language summary>",
        "score": <relevance score between 0 and 1>,
        "relevant": <true|false>
        }}
        Dont answer with Code or anything else.
    """

    response = llm(prompt)
    print(response)
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        return {
            "summary": "Could not parse model response.",
            "score": 0.0,
            "relevant": False
        }


<>:32: SyntaxWarning: invalid escape sequence '\R'
<>:32: SyntaxWarning: invalid escape sequence '\R'
/var/folders/33/gv0gjqf16cqcwn3y_4pclk_c0000gn/T/ipykernel_30844/4055476125.py:32: SyntaxWarning: invalid escape sequence '\R'
  full_description += "\n\Relationships:\n" + "\n".join(edge_descriptions)


In [29]:
def visualize_answer(result: List[PathResult]):
    import json
    G = nx.DiGraph()

    node_styles = {
        "Entity": "fill: #aed6f1",
        "Person": "fill: #f9e79f",
        "Organization": "fill: #d7bde2",
        "Location": "fill: #abebc6",
        "Unknown": "fill: #d5d8dc",
    }

    for path in result:
        for node in path.nodes:
            node_id = node.id
            if node_id not in G.nodes:
                properties = getattr(node, "properties", {})
                node_type = properties.get("type", next(iter(getattr(node, "labels", [])), "Unknown"))
                style = node_styles.get(node_type, "fill: #d5d8dc")

                G.add_node(
                    node_id,
                    node_type=node_type,
                    style=style,
                    **properties
                )

        for rel in path.relationships:
            source = rel.source
            target = rel.target
            rel_type = getattr(rel, "type", "rel")
            properties = getattr(rel, "properties", {})

            G.add_edge(
                source,
                target,
                label=rel_type,
                edge_type=rel_type,
                title=f"{rel_type}\n{json.dumps(properties, indent=2)}",
                **properties
            )

    graph_widget = GraphWidget(graph=G)
    display(graph_widget)


In [30]:
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    visualize_answer(structured_data)
    for data in structured_data:
        print(summarize_and_score_path(data, question))
    # unstructured coming soon
    final_data = f"""
        Structured data:
        {structured_data}
    """
    return final_data

In [31]:
print(retriever("What is the relationship between the Boss and Nadja Conti?"))

Search query: What is the relationship between the Boss and Nadja Conti?
{
  "names": ["Boss", "Nadja Conti"]
}


GraphWidget(layout=Layout(height='800px', width='100%'))

{
    "summary": "The Boss sent an email to Mrs. Money, but Nadja Conti is not mentioned in the path.",
    "score": 0.3,
    "relevant": false
}
{'summary': 'The Boss sent an email to Mrs. Money, but Nadja Conti is not mentioned in the path.', 'score': 0.3, 'relevant': False}
{
    "summary": "The Boss sent a message to Mrs. Money regarding funding allocations for tourism ventures, which is related to their colleague relationship.",
    "score": 0.8,
    "relevant": true
}
{'summary': 'The Boss sent a message to Mrs. Money regarding funding allocations for tourism ventures, which is related to their colleague relationship.', 'score': 0.8, 'relevant': True}
{
    "summary": "The Boss communicated with Nadja Conti, who is also known as Mrs. Money, about discussing funding allocations and meeting The Middleman.",
    "score": 0.8,
    "relevant": true
}
{'summary': 'The Boss communicated with Nadja Conti, who is also known as Mrs. Money, about discussing funding allocations and meeting T

/var/folders/33/gv0gjqf16cqcwn3y_4pclk_c0000gn/T/ipykernel_30844/4055476125.py:32: SyntaxWarning: invalid escape sequence '\R'
  full_description += "\n\Relationships:\n" + "\n".join(edge_descriptions)


KeyboardInterrupt: 

In [17]:
_template = """
    Given the following conversation and a follow-up question, rephrase the follow-up question to be a standalone question,
    in its original language.
    Chat History:
    {chat_history}
    Follow Up Input: {question}
    Standalone question:
"""
CONDENSE_QUESTION_PROMPT = PromptTemplate.from_template(_template)

def _format_chat_history(chat_history: List[Tuple[str, str]]) -> List:
    buffer = []
    for human, ai in chat_history:
        buffer.append(HumanMessage(content=human))
        buffer.append(AIMessage(content=ai))
    return buffer

_search_query = RunnableBranch(
    (
        RunnableLambda(lambda x: bool(x.get("chat_history"))).with_config(
            run_name="HasChatHistoryCheck"
        ),
        RunnablePassthrough.assign(
            chat_history=lambda x: _format_chat_history(x["chat_history"])
        )
        | CONDENSE_QUESTION_PROMPT
        | llm
        | StrOutputParser(),
    )
    , RunnableLambda(lambda x : x["question"]),
)

In [9]:
template = """
    Answer the question base only on the following context:
    {context}

    Question: {question}
    Use natural language and be concise.
    Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": retriever,
            "question": RunnablePassthrough(),
        }
    )
    |prompt
    |llm
    | StrOutputParser()
)


In [10]:
chain.invoke({"Who is Nadia Conti?"})

Search query: {'Who is Nadia Conti?'}
Here is the extracted information in JSON format:

{
  "names": ["Nadia Conti"]
}
Parsing failed: Expecting value: line 1 column 1 (char 0)


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

ModuleNotFoundError: No module named 'nl_querying'